# 音のプログラミング 第4回: フィルターと音色デザイン

**前回まで学んだこと：**
- 第1回：サイン波の生成、サンプリングの基礎
- 第2回：エンベロープ（ADSR）で音の時間変化をコントロール
- 第3回：FFTで音の周波数成分を分析

**今回の学習目標:**
- フィルターの仕組みを理解する
- ローパスフィルターで音色を変える
- 倍音構造と音色の関係を体験する
- シンプルな楽器音を作成する

**所要時間:** 90分

## 🛠️ 環境セットアップ

In [ ]:
# 🛠️ 環境セットアップ
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    !pip install japanize-matplotlib
    !git clone https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
    import japanize_matplotlib
else:
    print("🔧 ローカル環境を設定中...")
    sys.path.append('..')
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("✅ セットアップ完了")

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import (
    sine_wave, sawtooth_wave, square_wave, triangle_wave,
    adsr, AudioSignal, LowPassFilter,
)
from audio_lib.synthesis.note_utils import note_to_frequency, note_name_to_number
from audio_lib.notebook import play_sound
from scipy.fft import fft, fftfreq

## 🎯 実習1: フィルターとは何か？

フィルターは音の特定の周波数成分を削る（または強調する）効果です。

- **ローパスフィルター**：高い周波数をカット（低音は通す）
- **ハイパスフィルター**：低い周波数をカット（高音は通す）
- **バンドパスフィルター**：特定の周波数帯域のみ通す

今回は最も基本的な**ローパスフィルター**を学びます。

In [ ]:
# フィルターなしとありの音を比較してみよう
signal = sawtooth_wave(220, 1.0)

display(play_sound(signal, "元の音（フィルターなし — のこぎり波）"))

# ローパスフィルターを適用
lowpass = LowPassFilter(cutoff_freq=800)
filtered_signal = lowpass.process(signal)

display(play_sound(filtered_signal, "フィルター適用後（800Hz ローパス）"))

### 聞き比べてみよう
- フィルターなし：シャープで明るい音
- フィルターあり：柔らかくて暖かい音

フィルターによって高い周波数成分がカットされ、まろやかな音になりました。

## 🎯 実習2: カットオフ周波数の効果

In [ ]:
# 異なるカットオフ周波数を試してみよう
original_signal = sawtooth_wave(220, 1.0)

display(play_sound(original_signal, "元の音（フィルターなし）"))

for cutoff in [200, 500, 1000, 2000, 4000]:
    filter_lp = LowPassFilter(cutoff_freq=cutoff)
    filtered = filter_lp.process(original_signal)
    display(play_sound(filtered, f"カットオフ {cutoff}Hz"))

### スペクトラム分析：フィルターの効果を視覚化

In [ ]:
# フーリエ変換でスペクトラムを見てみよう
short_duration = 0.1
signal = sawtooth_wave(220, short_duration)

# フィルター適用
filter_lp = LowPassFilter(cutoff_freq=1000)
filtered_signal = filter_lp.process(signal)

# FFT計算
n = len(signal)
freqs = fftfreq(n, 1/signal.sample_rate)[:n//2]
original_fft = np.abs(fft(signal.data))[:n//2]
filtered_fft = np.abs(fft(filtered_signal.data))[:n//2]

# グラフ表示
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(freqs[:1000], original_fft[:1000], label='フィルターなし', alpha=0.7)
plt.plot(freqs[:1000], filtered_fft[:1000], label='ローパス(1000Hz)', alpha=0.7)
plt.axvline(x=1000, color='red', linestyle='--', label='カットオフ')
plt.xlabel('周波数 (Hz)')
plt.ylabel('振幅')
plt.title('周波数スペクトラム比較')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
time = np.linspace(0, short_duration, len(signal))
plt.plot(time[:1000], signal.data[:1000], label='フィルターなし', alpha=0.7)
plt.plot(time[:1000], filtered_signal.data[:1000], label='ローパス適用', alpha=0.7)
plt.xlabel('時間 (秒)')
plt.ylabel('振幅')
plt.title('波形比較（最初の0.02秒）')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 実習3: 楽器音の作成 — シンプルなピアノ風の音

In [ ]:
def create_piano_sound(note_freq, duration=2.0):
    """シンプルなピアノ風の音を作成"""
    # 基音と倍音を組み合わせ
    sig1 = sine_wave(note_freq, duration)
    sig2 = sine_wave(note_freq * 2, duration) * 0.3
    sig3 = sine_wave(note_freq * 3, duration) * 0.2
    sig4 = sine_wave(note_freq * 4, duration) * 0.1

    mixed = sig1 + sig2 + sig3 + sig4

    # ピアノ風のエンベロープ
    envelope = adsr(duration, attack=0.01, decay=0.3, sustain=0.4, release=1.5)
    final = mixed * envelope

    # ローパスフィルターで自然な響きに
    filter_lp = LowPassFilter(cutoff_freq=3000)
    return filter_lp.process(final)

# ピアノ風の音を試してみよう
for note in ['C4', 'E4', 'G4']:
    freq = note_to_frequency(note_name_to_number(note))
    sound = create_piano_sound(freq)
    display(play_sound(sound, f"ピアノ風 {note}"))

## 🎯 実習4: オルガン風の音を作ってみよう

In [ ]:
def create_organ_sound(note_freq, duration=3.0):
    """シンプルなオルガン風の音を作成"""
    sig1 = sine_wave(note_freq, duration) * 0.6
    sig2 = sine_wave(note_freq * 2, duration) * 0.4
    sig3 = sine_wave(note_freq * 3, duration) * 0.3

    mixed = sig1 + sig2 + sig3

    # オルガン風のエンベロープ（スローアタック、サスティン重視）
    envelope = adsr(duration, attack=0.5, decay=0.2, sustain=0.8, release=1.0)
    final = mixed * envelope

    filter_lp = LowPassFilter(cutoff_freq=4000)
    return filter_lp.process(final)

organ_c = create_organ_sound(note_to_frequency(note_name_to_number('C4')))
display(play_sound(organ_c, "オルガン風 C4"))

## 🎯 実習5: 自分だけの楽器音を作ろう

In [ ]:
def create_custom_instrument(note_freq, duration=2.0):
    """カスタム楽器音の作成 — パラメータを変更して実験してみよう！"""

    # ベース波形を選択（変更してみよう）
    # signal = sine_wave(note_freq, duration)      # 柔らかい
    signal = sawtooth_wave(note_freq, duration)    # シャープ
    # signal = square_wave(note_freq, duration)    # 電子的
    # signal = triangle_wave(note_freq, duration)  # 中間的

    # エンベロープパラメータ（変更してみよう）
    envelope = adsr(duration, attack=0.1, decay=0.5, sustain=0.6, release=1.0)
    final = signal * envelope

    # フィルターパラメータ（変更してみよう）
    filter_lp = LowPassFilter(cutoff_freq=2000)
    return filter_lp.process(final)

custom_sound = create_custom_instrument(note_to_frequency(note_name_to_number('C4')))
display(play_sound(custom_sound, "カスタム楽器音"))

実験してみよう：
- 波形を変更（sine_wave, sawtooth_wave, square_wave, triangle_wave）
- エンベロープ調整: attack, decay, sustain, release
- フィルター調整: cutoff_freq（500〜8000Hz）

## 🎯 実習6: 和音の作成 — 音楽的な響きを作ろう

In [ ]:
def create_chord_sound(note_frequencies, instrument_func, duration=3.0):
    """和音を作成"""
    chord_signal = None
    for freq in note_frequencies:
        note_signal = instrument_func(freq, duration)
        if chord_signal is None:
            chord_signal = note_signal
        else:
            chord_signal = chord_signal + note_signal
    return chord_signal * (1.0 / len(note_frequencies))

# Cメジャーコードの周波数
c_major_freqs = [
    note_to_frequency(note_name_to_number('C4')),
    note_to_frequency(note_name_to_number('E4')),
    note_to_frequency(note_name_to_number('G4')),
]

# ピアノ風で演奏
piano_chord = create_chord_sound(c_major_freqs, create_piano_sound)
display(play_sound(piano_chord, "Cメジャーコード（ピアノ風）"))

# オルガン風で演奏
organ_chord = create_chord_sound(c_major_freqs, create_organ_sound)
display(play_sound(organ_chord, "Cメジャーコード（オルガン風）"))

## 🏆 チャレンジ課題

### 課題1：フィルターのスイープ効果
時間とともにカットオフ周波数が変化する効果を作ってみよう

In [ ]:
def create_filter_sweep(note_freq, duration=4.0):
    """フィルタースイープ効果"""
    signal = sawtooth_wave(note_freq, duration)

    segment_length = len(signal) // 4
    cutoff_freqs = [300, 1000, 3000, 8000]

    filtered_data = np.zeros_like(signal.data)

    for i, cutoff in enumerate(cutoff_freqs):
        start = i * segment_length
        end = (i + 1) * segment_length if i < 3 else len(signal)

        segment = AudioSignal(signal.data[start:end], signal.sample_rate)
        filter_lp = LowPassFilter(cutoff_freq=cutoff)
        filtered_segment = filter_lp.process(segment)
        filtered_data[start:end] = filtered_segment.data

    return AudioSignal(filtered_data, signal.sample_rate)

sweep_sound = create_filter_sweep(note_to_frequency(note_name_to_number('C3')))
display(play_sound(sweep_sound, "フィルタースイープ効果（暗→明）"))

### 課題2：音楽フレーズの作成

In [ ]:
# 「きらきら星」の最初の部分をピアノ風の音で作ってみよう
def create_melody_note(note_name, duration=0.5):
    freq = note_to_frequency(note_name_to_number(note_name))
    return create_piano_sound(freq, duration)

notes = ['C4', 'C4', 'G4', 'G4', 'A4', 'A4', 'G4']
melody_signals = [create_melody_note(note).data for note in notes]

full_melody = AudioSignal(np.concatenate(melody_signals), 44100)
display(play_sound(full_melody, "きらきら星（最初の部分）"))

## 📚 今日のまとめ

### 学んだこと
1. **フィルター**の基本概念と効果
2. **ローパスフィルター**によるカットオフ周波数の調整
3. **楽器音の設計**：倍音構造とエンベロープの組み合わせ
4. **和音**の作成方法
5. **メロディー**の連結

### 次回予告
第5回では**オーディオエフェクト**を学びます。
- リバーブ、ディレイ、コーラス
- ダイナミクス（コンプレッサー、リミッター）
- より高度な音響効果